# 06 — Walk-Forward Backtest
Simulate the full trading strategy using walk-forward test predictions.
Includes trade management (partial TPs, breakeven stops, trailing).

In [ ]:
# Install dependencies
# torch, numpy, pandas are pre-installed by Colab — do NOT reinstall them.
# Pinning versions fights Colab's environment and causes resolver conflicts.
!pip install -q xgboost ccxt PyWavelets hmmlearn numba scikit-learn pyyaml \
    tqdm pyarrow matplotlib seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json

# Clone/update repo from GitHub (local Colab filesystem — fast)
REPO_DIR = '/content/scalp2_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone https://github.com/sergul74/Scalp2.git {REPO_DIR}

if not os.path.exists(os.path.join(REPO_DIR, 'scalp2', '__init__.py')):
    raise FileNotFoundError('scalp2 package not found after clone!')

sys.path.insert(0, REPO_DIR)

import logging
logging.basicConfig(level=logging.INFO)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scalp2.config import load_config
from scalp2.execution.trade_manager import TradeManager, TradeState, TradeStatus
from scalp2.utils.metrics import sharpe_ratio, sortino_ratio, max_drawdown, win_rate, profit_factor

config = load_config(f'{REPO_DIR}/config.yaml')
DATA_DIR = '/content/drive/MyDrive/scalp2/data/processed'

In [ ]:
# Load labeled dataset and walk-forward predictions
import pickle

df = pd.read_parquet(f'{DATA_DIR}/BTC_USDT_labeled.parquet')

with open(f'{DATA_DIR}/wf_predictions.pkl', 'rb') as f:
    wf_predictions = pickle.load(f)

print(f'Loaded {len(df)} bars, {len(wf_predictions)} walk-forward folds')

In [ ]:
# Walk-forward backtest with trade management + transaction costs + regime filters
from tqdm import tqdm

# Suppress per-trade log spam from TradeManager
logging.getLogger('scalp2.execution.trade_manager').setLevel(logging.WARNING)

seq_len = config.model.seq_len
exec_cfg = config.execution
trade_mgmt_cfg = config.execution.trade_management
label_cfg = config.labeling
order_cfg = config.execution.order_execution

trade_mgr = TradeManager(trade_mgmt_cfg, label_cfg.max_holding_bars)

# --- Transaction costs from config (Binance USDM Futures) ---
# order_type "limit" → maker fee; "market" → taker fee
if order_cfg.order_type == "limit":
    FEE_PER_SIDE_BPS = order_cfg.maker_fee_bps
else:
    FEE_PER_SIDE_BPS = order_cfg.taker_fee_bps
SLIPPAGE_PER_SIDE_BPS = order_cfg.slippage_bps
TOTAL_COST = 2 * (SLIPPAGE_PER_SIDE_BPS + FEE_PER_SIDE_BPS) / 10_000
print(f'TX cost model: {order_cfg.order_type} orders | '
      f'fee={FEE_PER_SIDE_BPS} bps/side + slip={SLIPPAGE_PER_SIDE_BPS} bps/side | '
      f'RT cost={TOTAL_COST*10000:.0f} bps')

# --- Precompute ADX and ATR percentile for filtering ---
MIN_ADX = exec_cfg.min_adx
MIN_ATR_PCTILE = exec_cfg.min_atr_percentile

# Rolling ATR percentile (96-bar window = ~24h on 15m)
if 'atr_14' in df.columns:
    df['atr_pctile'] = df['atr_14'].rolling(96, min_periods=10).rank(pct=True)
    df['atr_pctile'] = df['atr_pctile'].fillna(1.0)
else:
    df['atr_pctile'] = 1.0

print(f'Filters: min_adx={MIN_ADX}, min_atr_percentile={MIN_ATR_PCTILE}, '
      f'choppy_threshold={config.regime.choppy_threshold}')

all_trades = []
equity_curve = [0.0]
cumulative_pnl = 0.0
skip_reasons = {'low_adx': 0, 'low_volatility': 0, 'low_conf': 0,
                'hold': 0, 'daily_cap': 0, 'no_atr': 0}

for fold_data in tqdm(wf_predictions, desc='Backtesting folds'):
    fold_idx = fold_data['fold_idx']
    test_start = fold_data['test_start']
    test_end = fold_data['test_end']
    preds = fold_data['test_probabilities']   # (n_preds, 3)
    n_preds = len(preds)
    pred_offset = test_start + seq_len        # first bar with a prediction

    active = None
    entry_bar = None
    daily_count = 0
    prev_date = None

    for i in range(n_preds):
        bar = pred_offset + i
        if bar >= len(df):
            break

        row = df.iloc[bar]
        cur_date = row.name.date() if hasattr(row.name, 'date') else None
        if cur_date != prev_date:
            daily_count = 0
            prev_date = cur_date

        # ---- manage active trade ----
        if active is not None:
            active = trade_mgr.update(
                active, row['high'], row['low'], row['close']
            )
            if active.status not in (TradeStatus.OPEN, TradeStatus.PARTIAL_TP):
                net = active.pnl - TOTAL_COST
                cumulative_pnl += net
                all_trades.append(dict(
                    fold=fold_idx, direction=active.direction,
                    entry_price=active.entry_price,
                    bars_held=active.bars_held,
                    status=active.status.value,
                    gross_pnl=active.pnl, net_pnl=net,
                    entry_bar=entry_bar, exit_bar=bar,
                    timestamp=row.name,
                ))
                equity_curve.append(cumulative_pnl)
                active = None
            continue  # don't open new trade on same bar

        # ---- check for new signal ----
        p = preds[i]
        cls = int(np.argmax(p))
        if cls == 1:
            skip_reasons['hold'] += 1
            continue
        if max(p[0], p[2]) < exec_cfg.confidence_threshold:
            skip_reasons['low_conf'] += 1
            continue
        if daily_count >= exec_cfg.max_trades_per_day:
            skip_reasons['daily_cap'] += 1
            continue

        atr = row['atr_14'] if 'atr_14' in df.columns else 0.0
        if atr < 1e-10:
            skip_reasons['no_atr'] += 1
            continue

        # ADX filter (Faz 4.2)
        adx_val = row.get('adx_14', 999.0) if hasattr(row, 'get') else (
            row['adx_14'] if 'adx_14' in df.columns else 999.0)
        if adx_val < MIN_ADX:
            skip_reasons['low_adx'] += 1
            continue

        # ATR percentile filter (Faz 4.3)
        atr_pct = row['atr_pctile'] if 'atr_pctile' in df.columns else 1.0
        if atr_pct < MIN_ATR_PCTILE:
            skip_reasons['low_volatility'] += 1
            continue

        entry_price = row['close']
        if cls == 2:  # LONG
            sl = entry_price - label_cfg.sl_multiplier * atr
            direction = "LONG"
        else:         # SHORT
            sl = entry_price + label_cfg.sl_multiplier * atr
            direction = "SHORT"

        active = TradeState(
            direction=direction,
            entry_price=entry_price,
            current_stop_loss=sl,
            take_profit=0.0,       # managed dynamically by TradeManager
            atr_at_entry=atr,
        )
        entry_bar = bar
        daily_count += 1

    # force-close any open trade at fold boundary
    if active is not None:
        last_row = df.iloc[min(test_end - 1, len(df) - 1)]
        if active.direction == "LONG":
            unr = (last_row['close'] - active.entry_price) / active.entry_price
        else:
            unr = (active.entry_price - last_row['close']) / active.entry_price
        gross = active.pnl + unr * active.remaining_size
        net = gross - TOTAL_COST
        cumulative_pnl += net
        all_trades.append(dict(
            fold=fold_idx, direction=active.direction,
            entry_price=active.entry_price,
            bars_held=active.bars_held,
            status='CLOSED_FOLD_END',
            gross_pnl=gross, net_pnl=net,
            entry_bar=entry_bar,
            exit_bar=min(test_end - 1, len(df) - 1),
            timestamp=last_row.name,
        ))
        equity_curve.append(cumulative_pnl)
        active = None

trades_df = pd.DataFrame(all_trades)
print(f'\nTotal trades: {len(trades_df)}')
print(f'Cumulative net PnL: {cumulative_pnl*100:.2f}%')
print(f'\nSkip reasons: {skip_reasons}')

In [ ]:
# Results summary and visualization
if len(trades_df) == 0:
    print("No trades generated!")
else:
    net = trades_df['net_pnl'].values
    gross = trades_df['gross_pnl'].values
    n = len(trades_df)

    # Win/loss stats
    wins = net[net > 0]
    losses = net[net < 0]
    wr = len(wins) / n
    avg_win = wins.mean() if len(wins) else 0
    avg_loss = losses.mean() if len(losses) else 0
    pf = abs(wins.sum() / losses.sum()) if len(losses) else float('inf')

    # Daily P&L for Sharpe/Sortino
    trades_df['date'] = pd.to_datetime(trades_df['timestamp']).dt.date
    daily_pnl = trades_df.groupby('date')['net_pnl'].sum()
    full_range = pd.date_range(daily_pnl.index.min(), daily_pnl.index.max(), freq='D')
    daily_pnl = daily_pnl.reindex(full_range, fill_value=0.0)

    daily_sharpe = daily_pnl.mean() / (daily_pnl.std() + 1e-10) * np.sqrt(365)
    down = daily_pnl[daily_pnl < 0]
    daily_sortino = daily_pnl.mean() / (down.std() + 1e-10) * np.sqrt(365) if len(down) else 0

    # Max drawdown
    cum = np.cumsum(daily_pnl.values)
    peak = np.maximum.accumulate(cum)
    dd = peak - cum
    mdd = dd.max()
    calmar = (daily_pnl.mean() * 365) / mdd if mdd > 1e-10 else 0

    # Status distribution
    status_counts = trades_df['status'].value_counts()

    print('=' * 60)
    print('       WALK-FORWARD BACKTEST RESULTS')
    print('=' * 60)
    print(f'  Total trades       : {n}')
    print(f'  Win rate           : {wr:.4f} ({wr*100:.1f}%)')
    print(f'  Profit factor      : {pf:.4f}')
    print(f'  Expectancy/trade   : {net.mean()*100:.4f}%')
    print(f'  Avg win            : +{avg_win*100:.4f}%')
    print(f'  Avg loss           : {avg_loss*100:.4f}%')
    print(f'  Avg bars held      : {trades_df["bars_held"].mean():.1f}')
    print()
    print(f'  Daily Sharpe       : {daily_sharpe:.4f}')
    print(f'  Daily Sortino      : {daily_sortino:.4f}')
    print(f'  Max Drawdown       : {mdd*100:.4f}%')
    print(f'  Calmar             : {calmar:.4f}')
    print()
    print(f'  Gross PnL          : {gross.sum()*100:.2f}%')
    print(f'  Net PnL            : {net.sum()*100:.2f}%')
    print(f'  Cost impact        : {(gross.sum()-net.sum())*100:.2f}%')
    print(f'  TX cost/trade      : {TOTAL_COST*10000:.0f} bps')
    print()
    print('  Close reasons:')
    for status, count in status_counts.items():
        print(f'    {status:20s}: {count:5d} ({count/n*100:.1f}%)')
    print('=' * 60)

    # --- Charts ---
    fig, axes = plt.subplots(3, 1, figsize=(16, 12))

    # 1. Equity curve
    cum_pct = np.cumsum(daily_pnl.values) * 100
    axes[0].plot(daily_pnl.index, cum_pct, linewidth=1, color='#2196F3')
    axes[0].fill_between(daily_pnl.index, 0, cum_pct, alpha=0.1, color='#2196F3')
    axes[0].axhline(0, color='grey', linestyle='--', alpha=0.5)
    axes[0].set_title('Equity Curve (Cumulative PnL %)')
    axes[0].set_ylabel('Cumulative PnL (%)')
    axes[0].grid(True, alpha=0.3)

    # 2. Drawdown
    axes[1].fill_between(daily_pnl.index, 0, -dd * 100, alpha=0.4, color='red')
    axes[1].set_title('Drawdown')
    axes[1].set_ylabel('Drawdown (%)')
    axes[1].grid(True, alpha=0.3)

    # 3. PnL distribution
    axes[2].hist(net * 100, bins=50, alpha=0.7, color='#4CAF50', edgecolor='white')
    axes[2].axvline(0, color='red', linestyle='--', alpha=0.7)
    axes[2].set_title('Trade PnL Distribution')
    axes[2].set_xlabel('PnL per trade (%)')
    axes[2].set_ylabel('Count')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Monthly returns table
    trades_df['month'] = pd.to_datetime(trades_df['timestamp']).dt.to_period('M')
    monthly = trades_df.groupby('month')['net_pnl'].agg(['sum', 'count'])
    monthly.columns = ['return_pct', 'trades']
    monthly['return_pct'] *= 100
    print('\nMonthly Returns:')
    print(monthly.to_string())